In [2]:
import os
import json
import base64
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
PROJECT_ROOT = os.getcwd() 
DATASET_PATH = os.path.join(PROJECT_ROOT, "examples", "HarmfulMemes-tiny")
JSON_FILE = "train-tiny.jsonl"

In [ ]:
def load_top_n_samples(file_path, n=8):
    samples = [] 
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f): 
            if i >= n: 
                break
            samples.append(json.loads(line)) 
    return samples 

In [6]:
# Data Preprocessing
def get_multimodal_data(sample, root_path):
    text = sample['text']
    img_path = os.path.join(root_path, sample['img'])
    return text, img_path

In [7]:
# Image Processing
def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode('utf-8')

In [ ]:
# Message Construction
def build_multimodal_message(text, img_base64):
    """Constructs a multimodal message for LangChain."""
    return HumanMessage(content=[
        {
            "type": "text", 
            "text": f"Task: Analyze if the following social media post contains hateful speech.\n"
                    f"Text content: '{text}'\n"
                    f"Requirement: Consider both text and image context. Output <hateful> or <not hateful> with a brief explanation."
        },
        {
            "type": "image_url", 
            "image_url": {"url": f"data:image/jpeg;base64,{img_base64}"}
        }
    ])

In [14]:
# main

# samples = load_top_n_samples(json_path, 16)
chat = ChatOpenAI(
        model="gpt-4o", 
        temperature=0,
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url="https://api2.aigcbest.top/v1"
    )

json_path = os.path.join(DATASET_PATH, JSON_FILE)
samples = load_top_n_samples(json_path, 8)

for sample in samples:
    # 1. Extract Data
    text, path = get_multimodal_data(sample, DATASET_PATH)
    
    # 2. Processing images
    b64 = encode_image(path)
    
    # 3. Message
    msg = build_multimodal_message(text, b64)
    
    # 4. invoke
    res = chat.invoke([msg])
    print(res.content)

The post is "not hateful." The text emphasizes the importance of character over skin color, promoting a message of equality and non-discrimination. The images do not add any context that would change this interpretation.
The post is **not hateful**. The text encourages people to be open to love again and reassures them that not everyone will be like their past partners. The image of a couple in a wedding setting supports the positive and hopeful message about love and relationships.
The post is **not hateful**. 

The text "putting bows on your pet" is neutral and does not contain any language that is derogatory or harmful. The image shows a cat wearing a bow, which is a common and playful way to dress up pets. There is no indication of hate or malice in either the text or the image.
The post is "not hateful." The image shows people with rainbow flags, commonly associated with LGBTQ+ pride, and the text "they will soon be free" suggests a positive or hopeful message about liberation or 